# VideoTool cloud GPU whisper — Colab runner (BACKUP)

Same whisper-only core as the Kaggle runner, but I/O is via native `drive.mount`, so there
is no copy step — whisper reads the voice in place and writes the 2 output files back to the
same Drive folder. Use when the Kaggle weekly GPU quota is spent.

Enable a **GPU** runtime (Runtime → Change runtime type → T4 GPU). See
`docs/cloud-gpu-whisper-setup.md`.

**Drive caches (one-time, reused every later session):** the `large-v3` model (~3GB) is
downloaded once into `gdrive:_VIDEOTOOL_SHARED/models/large-v3` and the faster-whisper pip
wheels into `gdrive:_VIDEOTOOL_SHARED/wheelhouse/`. First run pays the download; every later
run loads the model straight from Drive and installs from the cached wheels — no re-download.

## 1. Mount Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Fetch cloud core + install videotool


In [ ]:
import shutil
SHARED = '/content/drive/MyDrive/_VIDEOTOOL_SHARED'
WHEELHOUSE = f'{SHARED}/wheelhouse'
shutil.copy(f'{SHARED}/videotool_cloud.py', '.')
import videotool_cloud as vc
vc.setup(wheelhouse=WHEELHOUSE)  # first run installs + caches wheels; later runs reuse them
import torch; assert torch.cuda.is_available(), 'No GPU — pick a T4 runtime.'

## 3. Run whisper on GPU (`large-v3`, float16)

Set `JOB_DIR` to the mounted Drive folder. For a shared drive use
`/content/drive/Shareddrives/...` instead of `MyDrive`.


In [ ]:
JOB_DIR = '/content/drive/MyDrive/1. YOUTUBE AUDIO/.../CHAP N'  # <-- EDIT THIS
MODEL_CACHE = '/content/drive/MyDrive/_VIDEOTOOL_SHARED/models'
caps, chaps = vc.run_whisper(
    JOB_DIR, model='large-v3', device='cuda', compute_type='float16',
    model_cache_dir=MODEL_CACHE,
)
print('captions:', caps)
print('chapters:', chaps)

## 4. Confirm outputs


In [ ]:
import os
print(os.listdir(os.path.join(JOB_DIR,'outputs')))
print(open(caps).read()[:500])